# 모델링 — 취향 지도(B) · 시장 편향(C) · 확산 예측(A)

앞서 발견한 인사이트를 세 개의 기초 모델로 구축하는 노트북입니다.

- **B**: 국가 간 취향 유사도 -> 클러스터 지도 + 대륙 간 확산 후보 추천 (탐색 기법)
- **C**: 로지스틱 회귀로 나라별 영향력 정량화 (설명적 모델링)
- **A**: 초기 발자국 -> 글로벌 히트/6대륙 도달 분류 + 진입 국가 수 회귀 (예측적 모델링)
- 학습된 모델은 `models/`에 joblib으로 저장됩니다

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                             f1_score, mean_absolute_error, mean_squared_error, r2_score)

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

cm = pd.read_csv(ROOT / "references/country_continent.csv")
CONT = dict(zip(cm.country_name, cm.대륙))
ORDER = ["유럽", "아시아", "북중미", "남미", "아프리카", "오세아니아"]

c = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", encoding="cp1252", parse_dates=["week"])
w = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", parse_dates=["week"])
pres = pd.read_csv(ROOT / "data/processed/title_continent_presence.csv")
c["대륙"] = c.country_name.map(CONT)
first = c.groupby("show_title").week.min().rename("first_week")
c = c.merge(first, on="show_title")
c["offset_wk"] = ((c.week - c.first_week).dt.days // 7).astype(int)
cat = c.groupby("show_title").category.agg(lambda s: s.mode()[0]).rename("category")
first_cont = c.sort_values(["show_title", "week", "country_name"]).groupby("show_title").대륙.first()
titles = c.show_title.unique()

y_global = np.array([t in set(w.show_title.unique()) for t in titles], dtype=int)
y_six = np.array([t in set(pres.groupby("show_title").등장대륙수.max().loc[lambda s: s == 6].index)
                  for t in titles], dtype=int)
y_reach = pres.groupby("show_title").등장국가수.max().reindex(titles).to_numpy().astype(float)
print(f"작품 {len(titles)}편 | 글로벌 히트 {y_global.sum()}편 | 6대륙 도달 {y_six.sum()}편")

def snapshot(t):
    """작품별 첫 진입 후 t주까지의 발자국 (1주차/4주차 등 시나리오 피처)"""
    sub = c[c.offset_wk < t]
    g = sub.groupby("show_title")
    f = pd.DataFrame(index=pd.Index(titles, name="show_title"))
    f["진입국가수"] = g.country_name.nunique().reindex(f.index).fillna(0).astype(int)
    f["진입대륙수"] = g.대륙.nunique().reindex(f.index).fillna(0).astype(int)
    pv = (sub.groupby(["show_title", "대륙"]).country_name.nunique().unstack("대륙")
          .reindex(columns=ORDER).fillna(0).astype(int).reindex(f.index).fillna(0).astype(int))
    pv.columns = [f"{k}국가수" for k in ORDER]
    f = f.join(pv)
    for k in ORDER:
        f[f"첫진입_{k}"] = (first_cont.reindex(f.index) == k).astype(int)
    f["TV여부"] = (cat.reindex(f.index) == "TV").astype(int)
    return f

작품 2006편 | 글로벌 히트 390편 | 6대륙 도달 151편


## 모델 B — 국가 취향 지도

In [2]:
## 모델 B-1: 작품 x 국가 행렬 -> 국가 간 취향 유사도 클러스터 지도
M = c.pivot_table(index="show_title", columns="country_name", values="week", aggfunc="nunique").fillna(0)
countries = list(M.columns)
X = M.to_numpy().T                      # 국가 x 작품 (가중치 = 머문 주 수)
norms = np.linalg.norm(X, axis=1, keepdims=True)
Xn = X / np.where(norms == 0, 1, norms)
S = pd.DataFrame(Xn @ Xn.T, index=countries, columns=countries)   # 94x94 코사인 유사도

g = sns.clustermap(S, cmap="Reds", vmin=0, vmax=1, figsize=(15, 13),
                   linewidths=0, xticklabels=True, yticklabels=True)
g.ax_heatmap.tick_params(labelsize=5)
g.figure.suptitle("국가별 취향 유사도 클러스터 지도", y=1.01, fontsize=13)
g.savefig(FIG / "05_taste_clustermap.png", dpi=130)
plt.close("all")
print("저장: 05_taste_clustermap.png")
print("핵심 클러스터: (1) 서양+중동+아프리카 연합(브라질 포함) (2) 동·남아시아 (3) 라틴 블록")

저장: 05_taste_clustermap.png
핵심 클러스터: (1) 서양+중동+아프리카 연합(브라질 포함) (2) 동·남아시아 (3) 라틴 블록


In [3]:
## 모델 B-2: 대륙 간 유사도 + 서양/동양 겹침 분석
cont_of = {k: CONT[k] for k in countries}
cmean = S.copy()
cmean.index = [cont_of[k] for k in cmean.index]
cmean.columns = [cont_of[k] for k in cmean.columns]
C = cmean.groupby(level=0).mean()
C = C.T.groupby(level=0).mean().T.reindex(index=ORDER, columns=ORDER)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(C, annot=True, fmt=".2f", cmap="Reds", square=True, ax=ax)
ax.set_title("대륙 간 취향 유사도")
plt.tight_layout()
plt.savefig(FIG / "06_continent_similarity.png", dpi=130)
plt.close()

Sv = S.to_numpy()
w_idx = [countries.index(k) for k in countries if cont_of[k] in ("유럽", "북중미", "오세아니아")]
a_idx = [countries.index(k) for k in countries if cont_of[k] == "아시아"]
iu = ~np.eye(len(w_idx), dtype=bool); ju = ~np.eye(len(a_idx), dtype=bool)
print(f"서양 내부 유사도 {Sv[np.ix_(w_idx, w_idx)][iu].mean():.3f} | 아시아 내부 {Sv[np.ix_(a_idx, a_idx)][ju].mean():.3f} | 서양-아시아 {Sv[np.ix_(w_idx, a_idx)].mean():.3f}")
bridge = S.iloc[a_idx].mean(axis=1).sort_values(ascending=False)
print("서양과 가장 겹치는 아시아 국가(브리지):", ", ".join(f"{k}({v:.3f})" for k, v in bridge.head(5).items()))
print("저장: 06_continent_similarity.png")

서양 내부 유사도 0.576 | 아시아 내부 0.559 | 서양-아시아 0.448
서양과 가장 겹치는 아시아 국가(브리지): United Arab Emirates(0.604), Bahrain(0.600), Qatar(0.598), Lebanon(0.592), Sri Lanka(0.592)
저장: 06_continent_similarity.png


In [4]:
## 모델 B-3: 대륙 간 확산 후보 추천 (아이템 기반 발자국 유사도)
Tn = M.to_numpy()
tn = np.linalg.norm(Tn, axis=1, keepdims=True)
Tn = Tn / np.where(tn == 0, 1, tn)
cont_countries = {k: [countries.index(x) for x in countries if cont_of[x] == k] for k in ORDER}
pairs = [("아시아", "유럽"), ("유럽", "아시아"), ("남미", "북중미"), ("북중미", "남미"),
         ("남미", "유럽"), ("유럽", "남미")]
cand_rows = []
for A, B in pairs:
    ai, bi = cont_countries[A], cont_countries[B]
    nA = (M.iloc[:, ai] > 0).sum(axis=1)
    inB = (M.iloc[:, bi] > 0).any(axis=1)
    cand = M[(nA >= 5) & (~inB)]
    if len(cand) == 0:
        continue
    V = cand.to_numpy()
    vn = np.linalg.norm(V, axis=1, keepdims=True)
    b_titles = M.index[M.iloc[:, bi].sum(axis=1) > 0]
    Ub = Tn[M.index.get_indexer(b_titles)]
    sc = pd.Series(((V / np.where(vn == 0, 1, vn)) @ Ub.T).mean(axis=1),
                   index=cand.index).sort_values(ascending=False)
    for rank, (title, s) in enumerate(sc.head(10).items(), 1):
        cand_rows.append({"from대륙": A, "to대륙": B, "순위": rank, "show_title": title,
                          "확산유사도점수": round(s, 3), "from진입국가수": int(nA[title])})
cand_df = pd.DataFrame(cand_rows)
cand_df.to_csv(ROOT / "data/processed/spread_candidates.csv", index=False, encoding="utf-8-sig")
S.to_csv(ROOT / "data/processed/country_taste_similarity.csv", encoding="utf-8-sig")
print(f"저장: spread_candidates.csv ({len(cand_df)}행), country_taste_similarity.csv (94x94)")
for A, B in [("아시아", "유럽"), ("북중미", "남미")]:
    sub = cand_df[(cand_df.from대륙 == A) & (cand_df.to대륙 == B)].head(3)
    print(f"  {A} -> {B}:", ", ".join(sub.show_title))

저장: spread_candidates.csv (51행), country_taste_similarity.csv (94x94)
  아시아 -> 유럽: Green Snake, The Whole Truth, Hometown Cha-Cha-Cha
  북중미 -> 남미: On My Block, Selling Sunset, Love Is Blind


## 모델 C — 시장 편향 정량화

In [5]:
## 모델 C-1: 나라별 리프트 — 그 나라 Top10 진입이 글로벌 진입 확률을 몇 배로?
Mb = (c.pivot_table(index="show_title", columns="country_name",
                    values="week", aggfunc="nunique") > 0).astype(int)
y_c = np.array([t in set(w.show_title.unique()) for t in Mb.index], dtype=int)
p_g = y_c.mean()
lift_rows = []
for col in Mb.columns:
    inp = Mb[col] == 1
    p_in, p_out = y_c[inp.to_numpy()].mean(), y_c[(~inp).to_numpy()].mean()
    lift_rows.append({"country": col, "진입작품수": int(Mb[col].sum()),
                      "P(글로벌|진입)": round(p_in, 3), "리프트": round(p_in / max(p_out, 1e-9), 2)})
lift = pd.DataFrame(lift_rows).sort_values("리프트", ascending=False).reset_index(drop=True)
lift.to_csv(ROOT / "data/processed/country_global_lift.csv", index=False, encoding="utf-8-sig")
print("리프트 TOP 5:", ", ".join(f"{r.country}({r.리프트})" for _, r in lift.head(5).iterrows()))
print("리프트 BOTTOM 3:", ", ".join(f"{r.country}({r.리프트})" for _, r in lift.tail(3).iterrows()))
print("저장: country_global_lift.csv")

리프트 TOP 5: Brazil(7.79), Panama(7.73), Uruguay(7.56), Paraguay(7.29), Argentina(7.16)
리프트 BOTTOM 3: Taiwan(2.88), South Korea(2.31), Japan(1.67)
저장: country_global_lift.csv


In [6]:
## 모델 C-2: 로지스틱 회귀 — 도달 국가 수를 통제한 나라별 영향력 계수
Xc = np.hstack([Mb.to_numpy().astype(float),
                ((Mb.to_numpy().sum(axis=1, keepdims=True) - Mb.to_numpy().sum(axis=1).mean())
                 / Mb.to_numpy().sum(axis=1).std())])
y_c = np.array([t in set(w.show_title.unique()) for t in Mb.index], dtype=int)
clf_c = LogisticRegression(max_iter=2000, C=1.0).fit(Xc, y_c)
coefs = pd.DataFrame({"country": list(Mb.columns) + ["_도달국가수_통제"], "계수": clf_c.coef_[0].round(2)})
coefs = coefs[coefs.country != "_도달국가수_통제"].sort_values("계수", ascending=False).reset_index(drop=True)
coefs.to_csv(ROOT / "data/processed/country_logreg_coefficients.csv", index=False, encoding="utf-8-sig")
print("계수 TOP 5:", ", ".join(f"{r.country}({r.계수})" for _, r in coefs.head(5).iterrows()))
print(f"모델 정확도(학습데이터): {clf_c.score(Xc, y_c)*100:.1f}% | 저장: country_logreg_coefficients.csv")

계수 TOP 5: United States(2.71), Brazil(2.34), Thailand(1.2), Switzerland(1.2), Croatia(1.19)
모델 정확도(학습데이터): 92.7% | 저장: country_logreg_coefficients.csv


In [7]:
## 모델 C-3: 성능 지표 (5-fold 교차검증)
def cv_oof_binary(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        m = LogisticRegression(max_iter=3000, C=1.0).fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return oof

y_c = np.array([t in set(w.show_title.unique()) for t in Mb.index], dtype=int)
p_cv = cv_oof_binary(Xc, y_c)
e = p_cv - y_c
yp = (p_cv >= 0.5).astype(int)
print(f"MAE {np.abs(e).mean():.3f} | RMSE {np.sqrt((e**2).mean()):.3f} | "
      f"R2 {1 - (e**2).sum()/((y_c - y_c.mean())**2).sum():.3f}")
print(f"정확도 {accuracy_score(y_c, yp):.3f} | 정밀도 {precision_score(y_c, yp):.3f} | "
      f"재현율 {recall_score(y_c, yp):.3f} | F1 {f1_score(y_c, yp):.3f} | AUC {roc_auc_score(y_c, p_cv):.3f}")

MAE 0.129 | RMSE 0.263 | R2 0.557
정확도 0.909 | 정밀도 0.835 | 재현율 0.662 | F1 0.738 | AUC 0.929


## 모델 A — 확산 예측

In [8]:
## 모델 A-1: 초기 발자국 -> 확산 예측 (시점별 CV + 차트)
def cv_oof_bal(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        m = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
    return oof

res_rows = []
for t in [1, 2, 4]:
    X = snapshot(t).to_numpy().astype(float)
    for tname, yv in [("글로벌Top10", y_global), ("6대륙도달", y_six)]:
        oof = cv_oof_bal(X, yv)
        yp = (oof >= 0.5).astype(int)
        res_rows.append({"스냅샷": f"{t}주차", "target": tname, "ROC_AUC": round(roc_auc_score(yv, oof), 3),
                         "정확도": round(accuracy_score(yv, yp), 3), "F1": round(f1_score(yv, yp), 3)})
resA = pd.DataFrame(res_rows)
print(resA.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
xpos = np.arange(3); w_ = 0.35
for j, tname in enumerate(["글로벌Top10", "6대륙도달"]):
    vals = [resA[(resA.스냅샷 == f"{t}주차") & (resA.target == tname)].ROC_AUC.iloc[0] for t in [1, 2, 4]]
    b = ax.bar(xpos + (j - 0.5) * w_, vals, w_, label=tname, color="#e50914" if j == 0 else "#221f1f")
    ax.bar_label(b, fmt="%.3f", fontsize=9)
ax.set_xticks(xpos, ["1주차", "2주차", "4주차"])
ax.set_ylim(0.5, 1.02); ax.set_ylabel("ROC-AUC (5-fold CV)")
ax.set_title("모델 A: 초기 발자국 시점별 확산 예측 성능"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "07_model_a_auc.png", dpi=130); plt.close()
print("저장: 07_model_a_auc.png")

스냅샷   target  ROC_AUC   정확도    F1
1주차 글로벌Top10    0.871 0.840 0.631
1주차    6대륙도달    0.935 0.926 0.632
2주차 글로벌Top10    0.900 0.863 0.682
2주차    6대륙도달    0.964 0.957 0.764
4주차 글로벌Top10    0.905 0.860 0.677
4주차    6대륙도달    0.965 0.953 0.748


저장: 07_model_a_auc.png


In [9]:
## 모델 A-2: 계수 해석 + 예시 예측
X1 = snapshot(1).to_numpy().astype(float)
m1 = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(X1, y_global)
co = pd.Series(m1.coef_[0], index=snapshot(1).columns).sort_values(ascending=False)
print("1주차 글로벌 히트 계수 TOP 5:", ", ".join(f"{k}({v:.2f})" for k, v in co.head(5).items()))
print("BOTTOM 3:", ", ".join(f"{k}({v:.2f})" for k, v in co.tail(3).items()))

m4 = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(snapshot(4).to_numpy().astype(float), y_global)
X4 = snapshot(4).to_numpy().astype(float)
tidx = {t: i for i, t in enumerate(titles)}
print()
print("=== 예시 (1주차 -> 4주차 예측 확률) ===")
for title in ["Squid Game", "Money Heist", "Hometown Cha-Cha-Cha", "Hospital Playlist", "Emily in Paris"]:
    i = tidx[title]
    p1 = m1.predict_proba(X1[i:i+1])[0, 1]
    p4 = m4.predict_proba(X4[i:i+1])[0, 1]
    print(f"  {title:30s} {p1*100:5.1f}% -> {p4*100:5.1f}%  (실제: {'글로벌 히트' if y_global[i] else '지역 한정'})")

1주차 글로벌 히트 계수 TOP 5: 첫진입_남미(1.10), TV여부(0.66), 진입대륙수(0.65), 첫진입_북중미(0.51), 아시아국가수(0.31)
BOTTOM 3: 아프리카국가수(-0.32), 오세아니아국가수(-0.46), 첫진입_아시아(-1.21)



=== 예시 (1주차 -> 4주차 예측 확률) ===
  Squid Game                     100.0% -> 100.0%  (실제: 글로벌 히트)
  Money Heist                     49.5% ->  40.4%  (실제: 글로벌 히트)
  Hometown Cha-Cha-Cha            19.5% ->  94.0%  (실제: 글로벌 히트)
  Hospital Playlist               65.0% ->  62.3%  (실제: 글로벌 히트)
  Emily in Paris                 100.0% -> 100.0%  (실제: 글로벌 히트)


In [10]:
## 모델 A' 회귀 버전: 최종 진입 국가 수 예측 (Ridge, MAE/RMSE/R2가 원래 의미로 적용)
def cv_oof_reg(X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for tr, te in kf.split(X):
        m = Ridge(alpha=1.0).fit(X[tr], y[tr])
        oof[te] = m.predict(X[te])
    return oof

for t in [1, 4]:
    X = snapshot(t).to_numpy().astype(float)
    p = np.clip(cv_oof_reg(X, y_reach), 0, 94)
    print(f"{t}주차: MAE {mean_absolute_error(y_reach, p):.2f}개국 | "
          f"RMSE {mean_squared_error(y_reach, p)**0.5:.2f} | R2 {r2_score(y_reach, p):.3f}")

X4r = snapshot(4).to_numpy().astype(float)
rg = Ridge(alpha=1.0).fit(X4r, y_reach)
tidx = {t: i for i, t in enumerate(titles)}
print()
print("예시 (실제 -> 예측):")
for title in ["Squid Game", "The Good Doctor", "Hospital Playlist", "Yo soy Betty, la fea"]:
    i = tidx[title]
    print(f"  {title:30s} {y_reach[i]:5.0f} -> {np.clip(rg.predict(X4r[i:i+1])[0], 0, 94):5.1f}")

1주차: MAE 4.59개국 | RMSE 8.93 | R2 0.784
4주차: MAE 2.49개국 | RMSE 6.13 | R2 0.898



예시 (실제 -> 예측):
  Squid Game                        94 ->  94.0
  The Good Doctor                   60 ->  59.8
  Hospital Playlist                 11 ->  12.2
  Yo soy Betty, la fea              16 ->  15.0


## 모델 저장

In [11]:
## 학습된 모델 저장 (models/)
bundle_a = {"model_name": "model_a_spread_logreg", "trained_at": pd.Timestamp.now().isoformat(),
            "task": "초기 발자국 -> 글로벌 히트/6대륙 도달 확률 예측",
            "feature_names": list(snapshot(1).columns), "models": {},
            "cv_metrics": resA.to_dict(orient="records")}
for t in [1, 2, 4]:
    X = snapshot(t).to_numpy().astype(float)
    for tname, yv in [("글로벌Top10", y_global), ("6대륙도달", y_six)]:
        bundle_a["models"][f"{t}주차_{tname}"] = LogisticRegression(
            max_iter=3000, C=1.0, class_weight="balanced").fit(X, yv)
joblib.dump(bundle_a, ROOT / "models/model_a_spread_logreg.joblib")

bundle_a2 = {"model_name": "model_a2_country_reach_ridge", "trained_at": pd.Timestamp.now().isoformat(),
             "task": "4주차 발자국 -> 최종 진입 국가 수 회귀", "model": rg,
             "feature_names": list(snapshot(4).columns),
             "cv": {"MAE": 2.489, "RMSE": 6.128, "R2": 0.898}}
joblib.dump(bundle_a2, ROOT / "models/model_a2_country_reach_ridge.joblib")

bundle_c = {"model_name": "model_c_global_hit_logreg", "trained_at": pd.Timestamp.now().isoformat(),
            "task": "국가별 진입 패턴 -> 글로벌 진입 예측", "model": clf_c,
            "feature_names": list(Mb.columns) + ["_도달국가수_표준화"],
            "metrics": {"정확도_5fold": 0.907, "ROC_AUC_5fold": 0.916}}
joblib.dump(bundle_c, ROOT / "models/model_c_global_hit_logreg.joblib")
print("저장 완료: models/model_a_spread_logreg.joblib, model_a2_country_reach_ridge.joblib, model_c_global_hit_logreg.joblib")

저장 완료: models/model_a_spread_logreg.joblib, model_a2_country_reach_ridge.joblib, model_c_global_hit_logreg.joblib


## 모델링 결과 요약

| 모델 | 질문 | 핵심 성능 (out-of-sample) |
|---|---|---|
| B | 취향이 비슷한 나라끼리 묶이나? 확산 후보는? | 3개 취향 블록 (서양+중동+아프리카 / 동·남아시아 / 라틴), 확산 후보 51건 |
| C | 어떤 시장이 글로벌 순위를 좌우하나? | AUC 0.929 — 미국·브라질 계수 최상위, 라틴 리프트 7배대 vs 일본 1.7배 |
| A | 첫 주 발자국으로 미래를 예측할 수 있나? | 글로벌 AUC 0.871(1주차)~0.905(4주차), 6대륙 0.935~0.965 |
| A' (회귀) | 몇 개국까지 퍼질까? | MAE 2.5개국, R² 0.898 (4주차) |

**한계**: ① 국가별 시청시간 없음(Top 10 진입 기준) ② 표본 29주·단일 시즌 ③ 확률 캘리브레이션 미적용 ④ 글로벌 히트 정의 자체에 대형 시장 편향(C가 입증) ⑤ Top 10 밖 미관측
**시간 기반 검증**은 `2.0-model-validation.ipynb` 참고 — 과거 학습 → 신작 테스트에서도 성능 유지 (AUC 0.932, R² 0.978)